# 📊 EDA — Telco Customer Churn

**Goal:** understand the data before modeling — find dirty values, class imbalance, and the strongest churn drivers.

*Dataset: IBM Telco Customer Churn — 7,043 customers, 19 features + target.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/telco_churn.csv')
df.head()

In [ ]:
df.info()
df.describe()

## 1. Target analysis — the classes are imbalanced

~73% of customers stayed, ~27% churned. Consequence: **accuracy is a misleading metric** here — a model that predicts 'no churn' for everyone scores 73% while being useless. We use **AUC** instead, and balanced class weights.

In [ ]:
counts = df['Churn'].value_counts()
print(counts / len(df))

ax = counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'],
                 title='Churn distribution')
ax.set_xticklabels(['Stayed', 'Churned'], rotation=0)

## 2. Churn rate by category

Which groups churn the most?

In [ ]:
cols = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, cols):
    rate = df.assign(churned=(df['Churn'] == 'Yes')).groupby(col)['churned'].mean()
    rate.plot(kind='bar', ax=ax, title=f'Churn rate by {col}')
plt.tight_layout()

**Findings:** month-to-month contracts churn far more than two-year contracts; fiber-optic customers churn more than DSL (interesting — usually the *better* product!); electronic-check payers churn the most.

## 3. Dirty data: TotalCharges is stored as text

In [ ]:
print(df['TotalCharges'].dtype)   # 'object' -> not numeric!

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Blank values found:', df['TotalCharges'].isna().sum())

The raw column is a string with ~11 blank entries (customers with 0 tenure). This is the classic silent data bug — models crash or silently misbehave. Fix: coerce to numeric, fill blanks with MonthlyCharges × tenure.

## 4. Numeric distributions by churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    df[df['Churn'] == 'No'][col].plot(kind='hist', ax=ax, alpha=0.5,
                                      density=True, label='Stayed')
    df[df['Churn'] == 'Yes'][col].plot(kind='hist', ax=ax, alpha=0.5,
                                       density=True, label='Churned')
    ax.set_title(col)
    ax.legend()

**Findings:** new customers (tenure < 6 months) churn heavily; high monthly charges correlate with churn.

## 5. Correlations

In [ ]:
num = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
num['churned'] = (df['Churn'] == 'Yes').astype(int)
corr = num.corr()

fig, ax = plt.subplots()
cax = ax.matshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)

## Summary — 5 insights that drive the model

1. **Contract type is the strongest driver** — month-to-month ≈ 43% churn vs ~3% for two-year.
2. **Tenure:** the first 6 months are the danger zone.
3. **Fiber-optic users churn more** — despite the better product.
4. **Electronic-check payment** correlates with churn.
5. **TotalCharges is dirty** (stored as strings) and collinear with tenure × MonthlyCharges — expected, not a problem for tree models.

These insights are later confirmed by SHAP in the final model — good EDA and good explanations should agree.